# 05 — Uncertainty Calibration

**The problem.** `greentruth/verdict.py` puts a 90% residual-bootstrap interval around
every observed change, and that interval now *decides* the verdict — when it spans
materially different outcomes, the system abstains. That makes the interval load-bearing.

But its coverage has never been measured. A "90% interval" that actually covers 60% of
the time would make the abstention rule arbitrary, and a "90%" label on it would be a
false claim. The README and the payload therefore say explicitly that it is **not** a
calibrated confidence interval.

**This notebook measures it**, and tests whether a conformal variant does better.

## What is measured

For every field and every ordered year pair `(y0, y1)` in the real series we can compute
the true fractional change. The question is whether an interval built from the *other*
fields' residual behaviour covers that true change at the advertised rate.

| method | description |
|---|---|
| **A. residual bootstrap** | what ships today, in `verdict.py` |
| **B. split conformal** | absolute-residual quantile, calibrated leave-one-field-out |
| **C. gap-conditional conformal** | separate quantile per year-gap, since error grows with horizon |

Evaluation is **leave-one-field-out**: calibrate on 11 fields, measure coverage on the
held-out 12th. Intervals are scored on empirical coverage and median width — a wide
interval trivially covers, so width is reported beside coverage everywhere.

## Honest limits, stated before the result

- **12 fields, 13 years.** That is 12 independent geographic units. Enough to *measure*
  coverage and expose a miscalibration; not enough to certify a tight guarantee.
- **Exchangeability is violated.** Conformal prediction assumes calibration and test
  points are exchangeable. These are autocorrelated annual series with trends, so the
  finite-sample guarantee is approximate here. Empirical coverage is the evidence; the
  theorem is not.
- If no method reaches nominal coverage, that is the finding, and GreenTruth keeps
  reporting an uncalibrated interval honestly rather than relabelling it.

| Notebook card | |
|---|---|
| **Type** | Evaluate uncertainty (no training) |
| **Purpose** | Measure the empirical coverage of the shipped residual-bootstrap interval, leave-one-field-out, and compare two conformal variants. |
| **Inputs** | The real per-field flaring series (downloaded from the public World Bank release). |
| **Outputs** | `05_calibration.json` → `notebooks/executed/results/`. |
| **Where it runs** | Google Colab or locally. |
| **Execution record** | Executed in Colab — record in `notebooks/executed/05_uncertainty_calibration.ipynb`. Its measurement is what the interface quotes per verdict. |


## 1. Real data — downloaded, not uploaded

In [ ]:
import io, json, itertools, os
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt

# Same public World Bank release notebook 03 uses. No account, no upload.
URL_SITES = ("https://thedocs.worldbank.org/en/doc/"
             "bd2432bbb0e514986f382f61b14b2608-0400072025/related/"
             "2012-2024-Flare-Volume-Estimates-by-individual-Flare-Location.xlsx")

FIELDS = {
    "us_permian":            dict(name="Permian Basin",             lat=31.9,  lon=-102.1, r=200),
    "us_bakken":             dict(name="Bakken",                    lat=47.8,  lon=-103.3, r=150),
    "russia_priobskoye":     dict(name="Priobskoye / West Siberia", lat=60.9,  lon=69.6,   r=150),
    "iraq_rumaila":          dict(name="Rumaila / Basra",           lat=30.05, lon=47.3,   r=90),
    "iran_south_pars":       dict(name="South Pars / Asaluyeh",     lat=27.5,  lon=52.6,   r=90),
    "algeria_hassi_messaoud":dict(name="Hassi Messaoud",            lat=31.67, lon=6.07,   r=60),
    "algeria_hassi_rmel":    dict(name="Hassi R'Mel",               lat=32.93, lon=3.28,   r=60),
    "nigeria_niger_delta":   dict(name="Niger Delta",               lat=5.3,   lon=6.0,    r=180),
    "venezuela_maracaibo":   dict(name="Lake Maracaibo",            lat=9.8,   lon=-71.5,  r=150),
    "libya_sirte":           dict(name="Sirte Basin",               lat=28.8,  lon=19.6,   r=180),
    "kazakhstan_tengiz":     dict(name="Tengiz",                    lat=46.1,  lon=53.5,   r=80),
    "mexico_cantarell":      dict(name="Cantarell / Campeche",      lat=19.4,  lon=-92.2,  r=120),
}

# Prefer a CSV already produced by notebook 03; otherwise rebuild it from source.
LOCAL = "flaring_by_field.csv"
for cand in (LOCAL, "data/real/flaring_by_field.csv", "../data/real/flaring_by_field.csv"):
    if os.path.exists(cand):
        LOCAL = cand
        break

if os.path.exists(LOCAL):
    df = pd.read_csv(LOCAL)
    print(f"using existing real series: {LOCAL}  ({len(df)} rows)")
else:
    print("downloading the public World Bank flare-location release…")
    r = requests.get(URL_SITES, timeout=300); r.raise_for_status()
    raw = pd.read_excel(io.BytesIO(r.content), sheet_name=0, header=None)
    hdr = next(i for i in range(min(12, len(raw)))
               if raw.iloc[i].astype(str).str.contains("Latitude", case=False).any())
    sites = pd.read_excel(io.BytesIO(r.content), sheet_name=0, header=hdr)
    sites.columns = [str(c).strip() for c in sites.columns]
    latc = next(c for c in sites.columns if "latitude" in c.lower())
    lonc = next(c for c in sites.columns if "longitude" in c.lower())
    ycols = {c: int(str(c)[:4]) for c in sites.columns
             if str(c)[:4].isdigit() and 2012 <= int(str(c)[:4]) <= 2024}

    def km(lat1, lon1, lat2, lon2):
        p = np.pi / 180
        a = (np.sin((lat2-lat1)*p/2)**2 +
             np.cos(lat1*p)*np.cos(lat2*p)*np.sin((lon2-lon1)*p/2)**2)
        return 12742 * np.arcsin(np.sqrt(a))

    lat = pd.to_numeric(sites[latc], errors="coerce").values
    lon = pd.to_numeric(sites[lonc], errors="coerce").values
    rows = []
    for fid, f in FIELDS.items():
        m = km(f["lat"], f["lon"], lat, lon) <= f["r"]
        for col, yr in ycols.items():
            v = pd.to_numeric(sites.loc[m, col], errors="coerce").sum()
            rows.append(dict(field=fid, year=yr, volume=float(v)))
    df = pd.DataFrame(rows)
    df.to_csv("flaring_by_field.csv", index=False)
    print("rebuilt from source ->", len(df), "rows")

series = {f: dict(zip(g.year.astype(int), g.volume.astype(float)))
          for f, g in df.groupby("field")}
print(f"\n{len(series)} fields, years "
      f"{min(min(s) for s in series.values())}-{max(max(s) for s in series.values())}")

## 2. The estimation task

For a field and a year pair `(y0, y1)`, the quantity GreenTruth reports is the fractional
change `(v1 - v0) / v0`. The point estimate is exact — it is arithmetic on two observed
values — so what needs an interval is not the *change* but the **uncertainty a reader
should attach to it** given how noisy that field's series is around its trend.

That is exactly what `verdict.py`'s residual bootstrap tries to express, so the honest
target for calibration is: *how far can the realised change at an unseen year pair sit
from the trend-implied change?* We measure that, per field, over all year pairs.

In [ ]:
def ols(xs, ys):
    n = len(xs); xb = sum(xs)/n; yb = sum(ys)/n
    sxx = sum((x-xb)**2 for x in xs)
    if sxx == 0: return yb, 0.0
    b1 = sum((x-xb)*(y-yb) for x, y in zip(xs, ys)) / sxx
    return yb - b1*xb, b1

def pairs_for(fid, min_gap=1):
    s = series[fid]; yrs = sorted(s)
    out = []
    for y0, y1 in itertools.combinations(yrs, 2):
        if y1 - y0 < min_gap or abs(s[y0]) < 1e-9:
            continue
        actual = (s[y1] - s[y0]) / s[y0]
        b0, b1 = ols(yrs, [s[y] for y in yrs])
        f0, f1 = b0 + b1*y0, b0 + b1*y1
        trend = (f1 - f0) / f0 if abs(f0) > 1e-9 else np.nan
        out.append(dict(field=fid, y0=y0, y1=y1, gap=y1-y0,
                        actual=actual, trend=trend,
                        resid=actual - trend if np.isfinite(trend) else np.nan))
    return out

allp = pd.DataFrame([r for f in series for r in pairs_for(f)]).dropna(subset=["resid"])
print(f"{len(allp)} (field, year-pair) observations across {allp.field.nunique()} fields")
print(allp.groupby("gap").resid.agg(["count", "mean", "std"]).round(3).head(12).to_string())
print("\nResidual = realised change minus trend-implied change.")
print("Its spread is what any honest interval has to cover.")

## 3. Three interval methods, leave-one-field-out

In [ ]:
import random
ALPHA = 0.10                     # nominal 90%

def bootstrap_interval(fid, y0, y1, level=0.90, n=2000, seed=7):
    '''Method A - exactly what greentruth/verdict.py does today.'''
    s = series[fid]; yrs = sorted(s); vals = [s[y] for y in yrs]
    b0, b1 = ols(yrs, vals)
    fitted = [b0 + b1*y for y in yrs]
    resid = [v - f for v, f in zip(vals, fitted)]
    i0, i1 = yrs.index(y0), yrs.index(y1)
    rng = random.Random(seed); ch = []
    for _ in range(n):
        vb = fitted[i0] + rng.choice(resid)
        vt = fitted[i1] + rng.choice(resid)
        if abs(vb) > 1e-12:
            ch.append((vt - vb) / vb)
    if not ch: return None
    ch.sort()
    lo_q, hi_q = (1-level)/2, 1-(1-level)/2
    return ch[int(lo_q*(len(ch)-1))], ch[int(hi_q*(len(ch)-1))]

def conformal_q(scores, alpha):
    '''Finite-sample-corrected conformal quantile.'''
    s = np.sort(np.asarray([x for x in scores if np.isfinite(x)]))
    if len(s) == 0: return np.nan
    k = int(np.ceil((len(s)+1)*(1-alpha)))
    return float(s[min(k, len(s)) - 1])

def evaluate(alpha=ALPHA):
    rows = []
    for held in series:
        cal = allp[allp.field != held]
        tst = allp[allp.field == held]
        q_global = conformal_q(cal.resid.abs(), alpha)
        q_by_gap = {g: conformal_q(gg.resid.abs(), alpha)
                    for g, gg in cal.groupby("gap")}
        for _, r in tst.iterrows():
            # A. bootstrap (what ships)
            bi = bootstrap_interval(held, int(r.y0), int(r.y1), 1-alpha)
            if bi:
                rows.append(dict(field=held, gap=r.gap, method="A. bootstrap (shipped)",
                                 covered=bi[0] <= r.actual <= bi[1],
                                 width=bi[1]-bi[0]))
            # B. split conformal, one global quantile
            rows.append(dict(field=held, gap=r.gap, method="B. split conformal",
                             covered=abs(r.resid) <= q_global, width=2*q_global))
            # C. conformal, one quantile per year-gap
            qg = q_by_gap.get(r.gap, q_global)
            rows.append(dict(field=held, gap=r.gap, method="C. gap-conditional conformal",
                             covered=abs(r.resid) <= qg, width=2*qg))
    return pd.DataFrame(rows)

ev = evaluate()
summary = (ev.groupby("method")
             .agg(coverage=("covered", "mean"),
                  median_width=("width", "median"),
                  n=("covered", "size"))
             .sort_values("coverage"))
summary["nominal"] = 1 - ALPHA
summary["gap_from_nominal"] = summary.coverage - (1 - ALPHA)
print("=" * 72)
print(f"EMPIRICAL COVERAGE, leave-one-field-out  (nominal {1-ALPHA:.0%})")
print("=" * 72)
print(summary.round(4).to_string())

## 4. Conditional coverage — where a single interval breaks

In [ ]:
bygap = (ev.groupby(["method", "gap"])
           .agg(coverage=("covered", "mean"), width=("width", "median"),
                n=("covered", "size")).reset_index())
piv = bygap.pivot(index="method", columns="gap", values="coverage")
print(f"coverage by year-gap (nominal {1-ALPHA:.0%}):")
print(piv.round(3).to_string())

dev = (piv - (1-ALPHA)).abs().mean(axis=1).sort_values()
print("\nmean |coverage - nominal| across gaps (lower = better conditional calibration):")
print(dev.round(4).to_string())

fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
for m, g in bygap.groupby("method"):
    g = g.sort_values("gap")
    ax[0].plot(g.gap, g.coverage, marker="o", label=m)
    ax[1].plot(g.gap, g.width, marker="o", label=m)
ax[0].axhline(1-ALPHA, color="r", ls="--", lw=1.4, label="nominal")
ax[0].set_xlabel("year gap"); ax[0].set_ylabel("empirical coverage")
ax[0].set_title("Conditional coverage"); ax[0].legend(fontsize=7)
ax[1].set_xlabel("year gap"); ax[1].set_ylabel("median interval width")
ax[1].set_title("Width — the price of coverage"); ax[1].legend(fontsize=7)

perfield = ev.groupby(["method", "field"]).covered.mean().reset_index()
for i, (m, g) in enumerate(perfield.groupby("method")):
    ax[2].scatter(g.covered, [i]*len(g), alpha=.7, label=m)
ax[2].axvline(1-ALPHA, color="r", ls="--", lw=1.4)
ax[2].set_yticks(range(perfield.method.nunique()))
ax[2].set_yticklabels([m[:14] for m in sorted(perfield.method.unique())], fontsize=7)
ax[2].set_xlabel("per-field coverage"); ax[2].set_title("Spread across the 12 fields")
plt.tight_layout(); plt.show()

## 5. Sweep the nominal level

In [ ]:
curve = []
for a in (0.20, 0.10, 0.05):
    e = evaluate(a)
    for m, g in e.groupby("method"):
        curve.append(dict(method=m, nominal=1-a, coverage=g.covered.mean(),
                          median_width=g.width.median()))
cv = pd.DataFrame(curve)
print(cv.pivot(index="method", columns="nominal", values="coverage").round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for m, g in cv.groupby("method"):
    g = g.sort_values("nominal")
    ax[0].plot(g.nominal, g.coverage, marker="o", label=m)
    ax[1].plot(g.nominal, g.median_width, marker="o", label=m)
ax[0].plot([0.78, 0.97], [0.78, 0.97], "k--", lw=1, label="perfect")
ax[0].set_xlabel("nominal"); ax[0].set_ylabel("empirical"); ax[0].set_title("Calibration curve")
ax[0].legend(fontsize=7)
ax[1].set_xlabel("nominal"); ax[1].set_ylabel("median width"); ax[1].set_title("Width")
plt.tight_layout(); plt.show()

## 6. Verdict and what to do with it

Read the printout below literally. If the shipped bootstrap is materially off nominal and
a conformal variant is closer, the concrete change is to add
`greentruth/calibration.py` holding the saved quantiles and have `verdict.py` use them —
**and only then** may the interface call the interval calibrated.

If nothing reaches nominal, GreenTruth keeps the honest uncalibrated label. With 12
fields that is a real possibility and it is not a failure of the notebook.

In [ ]:
best = summary.assign(absgap=summary.gap_from_nominal.abs()).sort_values("absgap")
shipped = summary.loc["A. bootstrap (shipped)"]
winner = best.index[0]
TOL = 0.05

print("=" * 72)
print("RESULT")
print("=" * 72)
print(f"shipped bootstrap coverage : {shipped.coverage:.3f} "
      f"(nominal {1-ALPHA:.2f}, off by {shipped.gap_from_nominal:+.3f})")
print(f"closest to nominal         : {winner} "
      f"({best.iloc[0].coverage:.3f}, width {best.iloc[0].median_width:.3f})")
print()
if abs(shipped.gap_from_nominal) <= TOL:
    print("-> The shipped interval is already within tolerance. No change needed;")
    print("   it may be described as empirically calibrated ON THIS DATA, with the")
    print("   exchangeability caveat retained.")
elif best.iloc[0].absgap <= TOL:
    print(f"-> Adopt {winner}. Save the quantiles below into")
    print("   greentruth/calibration.py and have verdict.py use them.")
else:
    print("-> NO method reached nominal coverage within tolerance.")
    print("   Report this as a negative result. GreenTruth must keep labelling its")
    print("   interval uncalibrated, which is what it does today. Do NOT relabel it.")

qs = {}
for a in (0.20, 0.10, 0.05):
    qs[str(a)] = dict(
        global_q=conformal_q(allp.resid.abs(), a),
        by_gap={int(g): conformal_q(gg.resid.abs(), a)
                for g, gg in allp.groupby("gap")})
out = dict(nominal_alpha=ALPHA, n_fields=int(allp.field.nunique()),
           n_observations=int(len(allp)),
           coverage=summary.reset_index().to_dict("records"),
           by_gap=bygap.to_dict("records"),
           level_sweep=cv.to_dict("records"),
           conformal_quantiles=qs,
           recommendation=winner if best.iloc[0].absgap <= TOL else "none_reached_nominal",
           caveats=[
               "12 fields = 12 independent geographic units. Enough to measure "
               "miscalibration, not to certify a tight guarantee.",
               "Exchangeability is violated: annual series are autocorrelated and "
               "trending, so the conformal guarantee is approximate here.",
               "Quantiles above are fitted on ALL fields; the coverage numbers come "
               "from leave-one-field-out, which is the honest estimate.",
           ])
with open("05_calibration.json", "w") as f:
    json.dump(out, f, indent=2, default=float)
print("\nsaved -> 05_calibration.json")